# 🚀 m0x-tune: Google Colab Auto-Launcher

Run all the cells below in order (1 by 1). This notebook will automatically clone the repository, install all dependencies, start both backend and frontend servers, and generate a secure public link for you to access the web UI.

## 1. Clone Repository & Setup Directory

In [ ]:
import os
import sys

# Clone repo if running on a fresh Colab instance
if not os.path.exists("M0x-tune") and not os.path.exists("scripts/install_deps.py"):
    print("Cloning m0x-tune repository...")
    !git clone https://github.com/M0X-Labs/M0x-tune.git
    %cd M0x-tune
elif os.path.exists("M0x-tune"):
    %cd M0x-tune

from pathlib import Path
PROJECT_ROOT = Path(os.getcwd()).resolve()
print(f"\nWorking directory: {PROJECT_ROOT}")

# Set up caching folders locally to avoid disk capacity constraints
os.environ["HF_HOME"] = str(PROJECT_ROOT / ".hf_home")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(PROJECT_ROOT / ".hf_home" / "hub")
os.environ["TEMP"] = str(PROJECT_ROOT / ".tmp")
os.environ["TMP"] = str(PROJECT_ROOT / ".tmp")

os.makedirs(".tmp", exist_ok=True)
os.makedirs(".hf_home/hub", exist_ok=True)

## 2. Install ML Stack & Web UI Dependencies

In [ ]:
print("Installing PyTorch, , and backend dependencies...")
!python scripts/install_deps.py

print("\nInstalling Next.js frontend dependencies...")
!cd finetune-ui && npm install

## 3. Start Backend & Frontend Servers

In [ ]:
import subprocess
import time

# Clean up previous logs
for log in ["backend.log", "frontend.log"]:
    if os.path.exists(log): os.remove(log)

# Start FastAPI backend (port 8000)
print("Starting FastAPI backend...")
backend_log = open("backend.log", "w")
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=backend_log,
    stderr=subprocess.STDOUT
)

# Start Next.js frontend (port 3000)
print("Starting Next.js frontend...")
frontend_log = open("frontend.log", "w")
frontend_proc = subprocess.Popen(
    ["npm", "run", "dev", "--", "-p", "3000"],
    cwd=str(PROJECT_ROOT / "finetune-ui"),
    stdout=frontend_log,
    stderr=subprocess.STDOUT
)

time.sleep(8)

# Validation
backend_ok = backend_proc.poll() is None
frontend_ok = frontend_proc.poll() is None

if backend_ok and frontend_ok:
    print("\n✅ Servers are successfully running in the background!")
else:
    if not backend_ok:
        print("❌ Backend failed to start. Logs:")
        with open("backend.log", "r") as f: print(f.read())
    if not frontend_ok:
        print("❌ Frontend failed to start. Logs:")
        with open("frontend.log", "r") as f: print(f.read())

## 4. Expose UI & Get Public URL

In [ ]:
import re
import urllib.request

cf_binary = PROJECT_ROOT / "cloudflared"
if not cf_binary.exists():
    print("Downloading cloudflared...")
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, cf_binary)
    cf_binary.chmod(0o755)

print("Starting tunnel to port 3000...")
tunnel_proc = subprocess.Popen(
    [str(cf_binary), "tunnel", "--url", "http://localhost:3000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Extract trycloudflare URL
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stderr.readline()
    if not line: break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "=" * 60)
    print("🎉 SUCCESS! The m0x-tune interface is live!")
    print(f"👉 CLICK TO OPEN: {tunnel_url}")
    print("=" * 60 + "\n")
    print("Keep this cell running to use the application.")
else:
    print("❌ Failed to extract Cloudflare tunnel URL. Cloudflare logs:")
    for _ in range(10):
        print(tunnel_proc.stderr.readline().strip())

## 5. Clean Up and Shutdown

In [ ]:
try:
    backend_proc.terminate()
    print("FastAPI backend stopped.")
except Exception: pass

try:
    frontend_proc.terminate()
    print("Next.js frontend stopped.")
except Exception: pass

try:
    tunnel_proc.terminate()
    print("Cloudflare Tunnel stopped.")
except Exception: pass

backend_log.close()
frontend_log.close()